In [1]:
from typing import Optional
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np
import pandas as pd
from sleepwalker.trainer.utils import cohen_kappa_from_confusion_matrix, read_jsonl
import math

def barplot_kappa_by_dataset(df, repeat:Optional[int] = None, title="Kappa by Model for Each Dataset"):
    """
    Create a 2xN subplot layout of bar plots.
    Each subplot corresponds to one dataset.
    X-axis: model
    Y-axis: kappa
    One colored bar per model.
    """
    if "repeat" in df.columns and repeat:
        df = df[df["repeat"] == repeat]

    datasets = sorted(df["dataset"].unique())
    n = len(datasets)

    cols = math.ceil(n / 2)
    rows = 2

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=datasets,
        horizontal_spacing=0.12,
        vertical_spacing=0.15,
    )

    colors = px.colors.qualitative.Set1
    model_list = sorted(df["model"].unique())
    color_map = {model: colors[i % len(colors)] for i, model in enumerate(model_list)}

    for i, dataset in enumerate(datasets):
        r = i // cols + 1
        c = i % cols + 1

        subset = df[df["dataset"] == dataset]
        show_legend_here = (i == 0)

        # Just one bar per model now — no repeat column anymore
        for model in model_list:
            mdata = subset[subset["model"] == model]
            if len(mdata) == 0:
                continue
            kappa_value = mdata["kappa"].values[0]

            fig.add_trace(
                go.Bar(
                    x=[model],
                    y=[kappa_value],
                    name=model,
                    legendgroup=model,
                    marker_color=color_map[model],
                    showlegend=show_legend_here,
                    text=[f"{kappa_value:.4f}"],
                    textposition="outside",
                ),
                row=r,
                col=c,
            )
        ymax = subset["kappa"].max()
        fig.update_yaxes(range=[0, ymax * 1.15], row=r, col=c)

    fig.update_layout(
        height=600,
        width=1200,
        barmode="group",
        title_text=title,
        showlegend=True,
    )

    return fig

def lineplot_kappa_by_dataset(df):
    """
    Create a 2xN subplot layout of line plots.
    Each subplot corresponds to one dataset.
    X-axis: repeat
    Y-axis: kappa
    One colored line per model.
    """
    datasets = sorted(df["dataset"].unique())
    n = len(datasets)

    # 2 rows, ceil(n / 2) columns
    cols = math.ceil(n / 2)
    rows = 2

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=datasets,
        horizontal_spacing=0.12,
        vertical_spacing=0.15,
    )

    colors = px.colors.qualitative.Set1
    model_list = sorted(df["model"].unique())
    color_map = {model: colors[i % len(colors)] for i, model in enumerate(model_list)}

    for i, dataset in enumerate(datasets):
        r = i // cols + 1
        c = i % cols + 1

        subset = df[df["dataset"] == dataset]
        show_legend_here = (i == 0)

        for model in model_list:
            mdata = subset[subset["model"] == model]
            if len(mdata) == 0:
                continue

            fig.add_trace(
                go.Scatter(
                    x=mdata["repeat"],
                    y=mdata["kappa"],
                    mode="lines+markers",
                    name=model,
                    legendgroup=model,
                    line=dict(color=color_map[model]),
                    showlegend=show_legend_here
                ),
                row=r,
                col=c,
            )

    fig.update_layout(
        height=600,
        width=1200,
        title_text="Kappa by Repeat for Each Dataset",
        showlegend=True,
    )

    return fig

def comparison_barplot_kappa_by_dataset(df, dff, name_df="df", name_dff="dff"):
    """
    Final version:
    - Correct legend (1 entry per model)
    - Correct colors + transparency
    - df/dff bars are close together
    - Clear separation between models
    - Spacing fully controllable
    """

    def hex_to_rgba(hex_color: str, alpha: float) -> str:
        hex_color = hex_color.lstrip("#")
        r = int(hex_color[0:2], 16)
        g = int(hex_color[2:4], 16)
        b = int(hex_color[4:6], 16)
        return f"rgba({r},{g},{b},{alpha})"

    datasets = sorted(set(df["dataset"]).union(set(dff["dataset"])))
    n_datasets = len(datasets)
    cols = math.ceil(n_datasets / 2)
    rows = 2

    fig = make_subplots(
        rows=rows, cols=cols, subplot_titles=datasets,
        horizontal_spacing=0.12, vertical_spacing=0.18,
    )

    model_list = sorted(set(df["model"]).union(set(dff["model"])))

    palette = px.colors.qualitative.Plotly
    model_colors = {m: palette[i % len(palette)] for i, m in enumerate(model_list)}

    max_kappa = max(df["kappa"].max(), dff["kappa"].max()) * 1.15

    # numeric positions per model
    base_x = np.arange(len(model_list))
    offset = 0.2         # distance between df and dff bars
    # distance between models => controlled by bar width + gaps

    for i, dataset in enumerate(datasets):
        r = i // cols + 1
        c = i % cols + 1

        subset1 = df[df["dataset"] == dataset].set_index("model")["kappa"].reindex(model_list)
        subset2 = dff[dff["dataset"] == dataset].set_index("model")["kappa"].reindex(model_list)

        df_x = base_x - offset
        dff_x = base_x + offset

        # df traces
        for j, model in enumerate(model_list):
            val_df = subset1.iloc[j]
            fig.add_trace(
                go.Bar(
                    x=[df_x[j]],
                    y=[None if np.isnan(val_df) else val_df],
                    name=model,
                    marker_color=model_colors[model],
                    showlegend=(i == 0),
                    legendgroup=model,
                    text=f"{val_df:.3f}" if not np.isnan(val_df) else "",
                    textposition="outside",
                ),
                row=r, col=c
            )

        # dff traces
        for j, model in enumerate(model_list):
            val_dff = subset2.iloc[j]
            fig.add_trace(
                go.Bar(
                    x=[dff_x[j]],
                    y=[None if np.isnan(val_dff) else val_dff],
                    name=None,
                    marker_color=hex_to_rgba(model_colors[model], 0.35),
                    showlegend=False,
                    legendgroup=model,
                    text=f"{val_dff:.3f}" if not np.isnan(val_dff) else "",
                    textposition="outside",
                ),
                row=r, col=c
            )

        # Model names centered between df and dff
        fig.update_xaxes(
            tickmode="array",
            tickvals=base_x,
            ticktext=model_list,
            row=r, col=c,
            tickangle=20
        )
    fig.update_layout(
        height=600,
        width=1200,
        barmode="overlay",   # overlay since positions already separate
        title_text=f"Cohens Kappa - {name_df} (links) vs. {name_dff} (rechts)",
        bargap=0.60,         # spacing *between models*
    )

    fig.update_yaxes(range=[0, max_kappa])
    return fig

/root/projects/sleepwalker/.venv/swa30/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np
import pandas as pd
from sleepwalker.trainer.utils import cohen_kappa_from_confusion_matrix, read_jsonl
import math

path = "../grouped-multiclass-foundation.jsonl"

df = read_jsonl(path)
entries_test = []
for d in df["dataset"].unique():
    for m in df["model"].unique():
        for r in df["repeat"].unique():
            cm = df.loc[(df["model"] == m) & (df["repeat"] == r) & (df["dataset"] == d), "test_cm"].values[0]
            kappa = cohen_kappa_from_confusion_matrix(cm)
            entries_test.append({"model":m, "dataset":d, "repeat":r, "kappa":kappa})
gdf_test = pd.DataFrame(entries_test)

entries_train = []
d = df["dataset"].unique()[0]
for m in df["model"].unique():
    best_model = df.loc[(df["model"] == m) & (df["repeat"] == 1) & (df["dataset"] == d), "best_model"].values[0]
    cm_train = df.loc[(df["model"] == m) & (df["repeat"] == 1) & (df["dataset"] == d), "train_cm"].values[0][best_model]["train"]
    cm_val = df.loc[(df["model"] == m) & (df["repeat"] == 1) & (df["dataset"] == d), "train_cm"].values[0][best_model]["val"]
    kappa_train = cohen_kappa_from_confusion_matrix(cm_train)
    kappa_val = cohen_kappa_from_confusion_matrix(cm_val)
    entries_train.append({"model":m, "kappa_train":kappa_train, "kappa_val":kappa_val})

gdf_train = pd.DataFrame(entries_train)
#gdf_train

fig = lineplot_kappa_by_dataset(gdf_test)
fig.show()

dff = gdf_test[gdf_test["repeat"] == gdf_test["repeat"].max()]
fig = barplot_kappa_by_dataset(gdf_test)
fig.show()


path = "../multiclass-foundation.jsonl"

df = read_jsonl(path)
entries_test = []
for d in df["dataset"].unique():
    for m in df["model"].unique():
        cm = df.loc[(df["model"] == m) & (df["dataset"] == d), "test_cm"].values[0]
        kappa = cohen_kappa_from_confusion_matrix(cm)
        entries_test.append({"model":m, "dataset":d, "kappa":kappa})
df_test_foundation = pd.DataFrame(entries_test)

fig = barplot_kappa_by_dataset(df_test_foundation, None, title="Cohens Kappa Foundation Models")
fig.show()

# dff = gdf_test[gdf_test["repeat"] == gdf_test["repeat"].max()]
# fig = comparison_barplot_kappa_by_dataset(dff, df_test, "grouped", "ungrouped")
# fig.show()

path = "../multiclass.jsonl"

df = read_jsonl(path)
entries_test = []
for d in df["dataset"].unique():
    for m in df["model"].unique():
        cm = df.loc[(df["model"] == m) & (df["dataset"] == d), "test_cm"].values[0]
        kappa = cohen_kappa_from_confusion_matrix(cm)
        entries_test.append({"model":m, "dataset":d, "kappa":kappa})
df_test = pd.DataFrame(entries_test)

fig = barplot_kappa_by_dataset(df_test, None, title="Cohens Kappa Regular Models")
fig.show()

fig = comparison_barplot_kappa_by_dataset(df_test_foundation, df_test, "Foundation", "Regulär")
fig.show()

# entries_train = []
# d = df["dataset"].unique()[0]
# for m in df["model"].unique():
#     best_model = df.loc[(df["model"] == m) & (df["dataset"] == d), "best_model"].values[0]
#     cm_train = df.loc[(df["model"] == m) & (df["dataset"] == d), "train_cm"].values[0][best_model]["train"]
#     cm_val = df.loc[(df["model"] == m) & (df["dataset"] == d), "train_cm"].values[0][best_model]["val"]
#     kappa_train = cohen_kappa_from_confusion_matrix(cm_train)
#     kappa_val = cohen_kappa_from_confusion_matrix(cm_val)
#     entries_train.append({"model":m, "kappa_train":kappa_train, "kappa_val":kappa_val})

# df_train = pd.DataFrame(entries_train)


#df_train
#dff_test


IndexError: index 0 is out of bounds for axis 0 with size 0